# C1-1.2 full run on Colab (CPU runtime)
Run the cells in order. After a disconnect: rerun cells 1, 2 (skip unzip if `/content/C1_v12` exists), 3, then 5 and 6; finished tasks are skipped.

In [ ]:
# 1. Mount Drive (completed tasks live on Drive; an unfinished task restarts)
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/C1_v12_run'
OUT = RUN + '/C1_out'
import os; os.makedirs(RUN, exist_ok=True); print(RUN)

In [ ]:
# 2. Unzip the uploaded package (upload C1_v12_upload.zip to /content or to RUN first)
import os, zipfile
src = '/content/C1_v12_upload.zip' if os.path.exists('/content/C1_v12_upload.zip') else RUN + '/C1_v12_upload.zip'
if not os.path.exists('/content/C1_v12'):
    zipfile.ZipFile(src).extractall('/content')
%cd /content/C1_v12
!ls

In [ ]:
# 3. Install and record the environment
!pip -q install -r requirements.txt
!python run_c1.py env

In [ ]:
# 4. Sanity + smoke (must end with SMOKE PASS; otherwise stop and report)
import subprocess
subprocess.run(["python", "run_c1.py", "smoke"], check=True)

In [ ]:
# 5. References in a background process (P4 budget cap 2400 s each)
import subprocess
if 'ref' not in globals() or ref.poll() is not None:
    ref_log = open(RUN + '/ref_stdout.log', 'a')
    ref = subprocess.Popen(['python', 'run_c1.py', 'run', '--group', 'ref', '--out', OUT], stdout=ref_log, stderr=subprocess.STDOUT)
print('ref pid', ref.pid)

In [ ]:
# 6. Main + sweep in the foreground (sequential timing). Resume = rerun this cell.
subprocess.run(['python', 'run_c1.py', 'run', '--group', 'nonref', '--out', OUT], check=True)
print("Waiting for reference process...")
assert ref.wait() == 0, "Reference run failed: report, do not patch."

In [ ]:
# 7. Progress / wait for references to finish
!python run_c1.py status --out {OUT}
print(open(RUN + "/ref_stdout.log").read()[-2000:])

In [ ]:
# 8. (optional) partial checkpoint now -> RUN/C1_partial_v12_latest.zip
!python run_c1.py checkpoint --out {OUT}

In [ ]:
# 9. Pack (only after status shows 938/938). Exit code 2 = missing tasks or strict plot failure: report it.
!python run_c1.py pack --out {OUT}; echo exit=$?
!cp -v /content/C1_v12/C1_results_v12_*.zip {RUN}/

In [ ]:
# 10. Download
import glob
from google.colab import files
z = sorted(glob.glob('/content/C1_v12/C1_results_v12_*.zip'))[-1]
print(z); files.download(z)